# N-HiTS: Architecture, Performance Evidence, and Application to AEMO Demand

## 1. Introduction

N-HiTS stands for **Neural Hierarchical Interpolation for Time Series Forecasting**. Challu et al. introduced it as an extension of N-BEATS for long-horizon forecasting. It addresses two problems that become more important as the forecast grows:

1. forecasts can become increasingly inaccurate or volatile; and
2. producing a separate full-resolution output for every future point can require a large model and substantial computation.

N-HiTS represents the forecast at several temporal resolutions. Rather than asking one component to learn the complete future signal at full detail, it combines coarse components for broad behaviour, intermediate components for medium-scale patterns, and fine components for short-term detail. The final prediction is the element-wise sum of these components.

Its three central techniques are **multi-rate input sampling**, **hierarchical interpolation**, and **doubly residual stacking**. Unlike an LSTM or Transformer, its core architecture consists mainly of multilayer perceptrons (MLPs).

![N-HiTS architecture showing its stacks, blocks, pooling and interpolation](public/nhits_architecture.png)

*Figure 1. Overall N-HiTS architecture, adapted from Challu et al. (2023).*

## 2. Forecasting problem

Let the historical input window be

$$
\mathbf y_{t-L+1:t}=[y_{t-L+1},\ldots,y_t],
$$

where $L$ is the look-back length. The objective is to predict the following $H$ observations:

$$
\hat{\mathbf y}_{t+1:t+H}=[\hat y_{t+1},\ldots,\hat y_{t+H}].
$$

N-HiTS learns

$$
f_{\boldsymbol\Theta}:\mathbf y_{t-L+1:t}
\longrightarrow \hat{\mathbf y}_{t+1:t+H}.
$$

It is a **direct multi-step model**: all $H$ forecast values are produced in one forward pass. Within this horizon, forecast step 2 is not recursively generated from forecast step 1. Every output is derived from the same historical input.

N-HiTS is therefore designed to make one potentially long, fixed-horizon prediction directly. It is not inherently designed to create an unlimited forecast by repeatedly feeding a short forecast back into itself.

## 3. Essential terminology

| Term | Meaning |
|---|---|
| Forecast | A prediction of future observations that are not yet known. |
| Backcast | A reconstruction of the known historical input produced inside a block. It is not a prediction into the past. |
| Residual | The portion of the historical input that previous blocks have not explained. |
| Block | A neural processing unit that produces one backcast and one forecast component. |
| Stack | One or more blocks operating at a selected temporal resolution. |
| Pooling | Compressing neighbouring observations to give a block a coarser view of history. |
| Interpolation | Expanding a small set of predicted coefficients, or knots, into a complete sequence. |

## 4. Overall processing flow

Each block receives the currently unexplained history and:

1. pools the input at its temporal resolution;
2. extracts features through an MLP;
3. generates backcast and forecast coefficients;
4. interpolates those coefficients into full sequences;
5. subtracts its backcast from the current history; and
6. adds its forecast component to the accumulated prediction.

```mermaid
flowchart TD
    A["Historical input"] --> B["Pool at one resolution"]
    B --> C["MLP feature extraction"]
    C --> D["Backcast and forecast coefficients"]
    D --> E["Interpolate full sequences"]
    E --> F["Subtract backcast"]
    E --> G["Add forecast component"]
    F --> H["Next block at a finer resolution"]
    H --> B
```

## 5. Architecture layer by layer

The worked examples in this section all use one deliberately tiny model — look-back $L=8$, horizon $H=4$, two blocks — with the same input throughout:

$$
\mathbf y_1 = [100,\ 120,\ 118,\ 90,\ 105,\ 125,\ 122,\ 95]\ \text{MW}.
$$

Block 1 is coarse (pool kernel $k_1=2$, ratio $r_1=0.5$); block 2 is fine ($k_2=1$, $r_2=1.0$). MLP and linear-head weights are learned, so where a real network would apply trained weights the example just states a representative output and its shape.

### 5.1 Input residual

The first block receives the original input:

$$
\mathbf y_1=\mathbf y_{t-L+1:t}.
$$

Later blocks receive residual inputs after earlier blocks remove the historical components they reconstructed.

*Worked example.* Block 1 receives $\mathbf y_1=[100,120,118,90,105,125,122,95]$. Block 2 will instead receive $\mathbf y_2=\mathbf y_1-\tilde{\mathbf y}_1$, the part block 1 could not rebuild (computed in 5.6).

### 5.2 Multi-rate input sampling

For block $\ell$, the paper applies pooling with kernel size $k_\ell$:

$$
\mathbf y^{(p)}_\ell
=\operatorname{MaxPool}(\mathbf y_\ell,k_\ell). \tag{1}
$$

A large kernel combines more neighbouring observations and creates a compressed view, helping the block concentrate on broad movement. A small kernel retains more local information.

| Resolution | Pooling kernel | Information emphasised |
|---|---:|---|
| Coarse | Large $k_\ell$ | Overall level and slow movement |
| Intermediate | Moderate $k_\ell$ | Medium-scale cycles |
| Fine | Small $k_\ell$ | High-frequency detail |

Pooling reduces the number of values processed by coarse MLPs while retaining coverage of the complete historical window.

*Worked example.* Block 1 pools $\mathbf y_1$ with $k_1=2$, taking the maximum of each non-overlapping pair:

$$
\max(100,120)=120,\quad
\max(118,90)=118,\quad
\max(105,125)=125,\quad
\max(122,95)=122,
$$

$$
\mathbf y^{(p)}_1=[120,\ 118,\ 125,\ 122].
$$

Eight values become four, so the coarse MLP processes half as many inputs while still spanning the whole window. Block 2 uses $k_2=1$: no pooling, $\mathbf y^{(p)}_2=\mathbf y_2$ unchanged (eight values).

### 5.3 Nonlinear feature extraction

The pooled input is passed into an MLP:

$$
\mathbf h_\ell=\operatorname{MLP}_\ell(\mathbf y^{(p)}_\ell). \tag{2a}
$$

Two linear heads produce coefficient vectors:

$$
\boldsymbol\theta^f_\ell
=\operatorname{LINEAR}^f_\ell(\mathbf h_\ell), \tag{2b}
$$

$$
\boldsymbol\theta^b_\ell
=\operatorname{LINEAR}^b_\ell(\mathbf h_\ell). \tag{2c}
$$

$\boldsymbol\theta^f_\ell$ controls the forecast, whereas $\boldsymbol\theta^b_\ell$ controls the backcast. These coefficients act as temporal control points rather than necessarily representing every timestamp directly.

*Worked example.* Block 1:

- $\mathbf h_1=\operatorname{MLP}_1([120,118,125,122])$ — a hidden vector of length 512 (learned weights, not shown). The MLP turns the four pooled values into features.
- Forecast head: $\boldsymbol\theta^f_1=\operatorname{LINEAR}^f_1(\mathbf h_1)=[110,\ 106]$ — two numbers, because block 1's ratio gives $\lceil 0.5\cdot 4\rceil=2$ knots (see 5.4).
- Backcast head: $\boldsymbol\theta^b_1=\operatorname{LINEAR}^b_1(\mathbf h_1)=[112,\ 110]$ — two knots for the length-8 backcast.

$[110,106]$ is not "the forecast for steps 1 and 2"; it is two control points that 5.4 expands into all four horizon values.

### 5.4 Hierarchical interpolation

The number of forecast coefficients is controlled by the block's expressiveness ratio $r_\ell$:

$$
|\boldsymbol\theta^f_\ell|=\lceil r_\ell H\rceil.
$$

A coarse block uses a small $r_\ell$, predicts relatively few knots, and interpolates them across the complete horizon. A fine block uses a larger $r_\ell$ and can represent more detailed movement.

The forecast and backcast components are reconstructed as

$$
\hat y_{\tau,\ell}=g(\tau,\boldsymbol\theta^f_\ell),
\qquad \tau\in\{t+1,\ldots,t+H\}, \tag{3a}
$$

$$
\tilde y_{\tau,\ell}=g(\tau,\boldsymbol\theta^b_\ell),
\qquad \tau\in\{t-L+1,\ldots,t\}. \tag{3b}
$$

The interpolation function $g$ may use nearest-neighbour, linear, or cubic interpolation. For linear interpolation between neighbouring coefficient locations $t_1$ and $t_2$,

$$
g(\tau,\boldsymbol\theta)
=\theta[t_1]
+\frac{\theta[t_2]-\theta[t_1]}{t_2-t_1}(\tau-t_1). \tag{4}
$$

For example, if $H=48$, a coarse block might predict only six coefficients and interpolate them across all 48 positions. A fine block may predict many more coefficients to represent individual peaks and troughs.

*Worked example — forecast component of block 1.* Knot count $\lceil r_1 H\rceil=\lceil 0.5\cdot 4\rceil=2$. Place the knots at the horizon ends: $\theta[1]=110$ at $\tau=1$, $\theta[4]=106$ at $\tau=4$. Apply (4) with $t_1=1,\ t_2=4$:

$$
g(2)=110+\frac{106-110}{4-1}(2-1)=110-1.33=108.67,
$$

$$
g(3)=110+\frac{106-110}{4-1}(3-1)=110-2.67=107.33,
$$

$$
\hat{\mathbf y}_{\cdot,1}=[110.00,\ 108.67,\ 107.33,\ 106.00].
$$

Two learned numbers became a smooth four-point line.

*Worked example — backcast of block 1.* $\boldsymbol\theta^b_1=[112,110]$, knots at positions 1 and 8, slope $(110-112)/(8-1)=-0.286$ per step:

$$
\tilde{\mathbf y}_1=[112.00,\ 111.71,\ 111.43,\ 111.14,\ 110.86,\ 110.57,\ 110.29,\ 110.00].
$$

For contrast, block 2 has $r_2=1.0$, so $\lceil 1.0\cdot 4\rceil=4$ knots — one per horizon step, and no interpolation is needed.

### 5.5 Coarse-to-fine specialisation

Pooling and interpolation are coordinated:

$$
k_{\text{coarse}}>k_{\text{fine}},
\qquad
r_{\text{coarse}}<r_{\text{fine}}.
$$

The resulting learned decomposition can be written conceptually as

$$
\text{forecast}
=\text{coarse component}
+\text{intermediate component}
+\text{fine component}.
$$

The model is not explicitly told that one stack must represent a trend and another a daily cycle. The division by temporal scale emerges through training.

*Worked example.* In the toy model $k_1=2>k_2=1$ (block 1 pools more, sees a coarser signal) and $r_1=0.5<r_2=1.0$ (block 1 has fewer knots, produces a smoother output). Block 1's forecast $[110.00,108.67,107.33,106.00]$ is a straight gently-declining line — the level and slow drift. Block 2, working on the residual with four knots, adds the sharp peak-and-trough shape (5.7).

### 5.6 Backcast residual connection

Each block subtracts its backcast from its input:

$$
\mathbf y_{\ell+1}=\mathbf y_\ell-\tilde{\mathbf y}_\ell.
$$

The next block therefore works on what earlier blocks failed to reconstruct.

*Worked example.* $\mathbf y_2=\mathbf y_1-\tilde{\mathbf y}_1$:

| position | $\mathbf y_1$ | $\tilde{\mathbf y}_1$ | $\mathbf y_2$ |
|---:|---:|---:|---:|
| 1 | 100 | 112.00 | −12.00 |
| 2 | 120 | 111.71 | 8.29 |
| 3 | 118 | 111.43 | 6.57 |
| 4 | 90 | 111.14 | −21.14 |
| 5 | 105 | 110.86 | −5.86 |
| 6 | 125 | 110.57 | 14.43 |
| 7 | 122 | 110.29 | 11.71 |
| 8 | 95 | 110.00 | −15.00 |

$\mathbf y_2$ is the oscillation around block 1's smooth line — exactly what block 1 could not capture. Block 2 pools this at $k_2=1$ and forecasts from it.

### 5.7 Forecast residual connection

The partial forecasts are added element by element:

$$
\hat{\mathbf y}_{t+1:t+H}
=\sum_{\ell=1}^{B}\hat{\mathbf y}_{t+1:t+H,\ell}.
$$

This explains the statement that **the final forecast is the sum of the components**.

*Worked example.* Block 1's forecast is $[110.00,108.67,107.33,106.00]$ from 5.4. Say block 2's forecast component (fine, derived from $\mathbf y_2$) is $[-8.0,\ 12.0,\ 9.0,\ -13.0]$. The final forecast is the sum:

| step $\tau$ | block 1 (coarse) | block 2 (fine) | final $\hat y$ |
|---:|---:|---:|---:|
| 1 | 110.00 | −8.0 | 102.00 |
| 2 | 108.67 | 12.0 | 120.67 |
| 3 | 107.33 | 9.0 | 116.33 |
| 4 | 106.00 | −13.0 | 93.00 |

The coarse block sets the level (~106–110 MW); the fine block restores the peak-and-trough shape.

A larger illustration with the same structure:

$$
\begin{aligned}
\text{coarse} &= [1500,1500,1500,1500],\\
\text{intermediate} &= [-100,0,200,100],\\
\text{fine} &= [20,-30,50,-10],\\
\text{final forecast} &= [1420,1470,1750,1590].
\end{aligned}
$$

The architecture is called **doubly residual** because a backward path subtracts backcasts from history while a forward path adds forecast components to the final output.

## 6. Training

Training samples are created by moving look-back and forecast windows through the training series:

$$
\mathbf y_{i-L+1:i}\longrightarrow\mathbf y_{i+1:i+H}.
$$

*Worked example.* With $L=8$, $H=4$ and a 16-point training series, slide both windows one step at a time:

| origin $i$ | input | target |
|---:|---|---|
| 8 | $y_1\ldots y_8$ | $y_9\ldots y_{12}$ |
| 9 | $y_2\ldots y_9$ | $y_{10}\ldots y_{13}$ |
| … | … | … |
| 12 | $y_5\ldots y_{12}$ | $y_{13}\ldots y_{16}$ |

A 16-point series yields $16-8-4+1=5$ windows. AEMO's 2018–2019 window ($\sim$35,000 half-hourly points, $L=672$, $H=336$) yields roughly 34,000 overlapping windows.

Typical objectives include mean absolute error and mean squared error:

$$
\operatorname{MAE}=\frac{1}{H}\sum_{j=1}^{H}|y_{t+j}-\hat y_{t+j}|,
$$

$$
\operatorname{MSE}=\frac{1}{H}\sum_{j=1}^{H}(y_{t+j}-\hat y_{t+j})^2.
$$

*Worked example.* Take the window whose target is $\mathbf y_{9:12}=[98,\ 122,\ 115,\ 92]$ and the forecast $\hat{\mathbf y}=[102.00,\ 120.67,\ 116.33,\ 93.00]$ from 5.7:

| step | $y$ | $\hat y$ | $|y-\hat y|$ | $(y-\hat y)^2$ |
|---:|---:|---:|---:|---:|
| 1 | 98 | 102.00 | 4.00 | 16.00 |
| 2 | 122 | 120.67 | 1.33 | 1.77 |
| 3 | 115 | 116.33 | 1.33 | 1.77 |
| 4 | 92 | 93.00 | 1.00 | 1.00 |

$$
\operatorname{MAE}=\frac{4.00+1.33+1.33+1.00}{4}=\frac{7.66}{4}=1.92\ \text{MW},
$$

$$
\operatorname{MSE}=\frac{16.00+1.77+1.77+1.00}{4}=\frac{20.54}{4}=5.14\ \text{MW}^2.
$$

The single 4 MW miss at step 1 is 4 of the 7.66 in the MAE sum but 16 of the 20.54 in the MSE sum, so training with MSE would push harder to remove that one large error.

Gradient-based optimisation updates the MLPs and coefficient heads so that their combined forecast has lower error across the training windows.

The paper expresses the multiresolution representation as

$$
Y(\tau\mid\mathbf y)
\approx
\sum_{w,h}\hat\theta_{w,h}(\mathbf y)\phi_{w,h}(\tau).
$$

This means a hierarchy of coarse and fine basis functions can efficiently approximate a sufficiently smooth future function. It describes representational capacity, not guaranteed accuracy over an unlimited future.

*Worked example.* Read 5.7's table as this sum. The coarse term $\hat\theta_{\text{coarse}}\,\phi_{\text{coarse}}(\tau)$ is the interpolation of $[110,106]$, giving $[110.00,108.67,107.33,106.00]$; the fine term $\hat\theta_{\text{fine}}\,\phi_{\text{fine}}(\tau)$ is $[-8.0,12.0,9.0,-13.0]$; their sum is the forecast $[102.00,120.67,116.33,93.00]$. The $\phi$ are fixed basis functions — here linear "tent" shapes between knot positions — and training learns only the $\hat\theta$ that scale them.

## 7. Why N-HiTS is a long-horizon model

N-HiTS can efficiently produce a comparatively large fixed $H$ in one direct pass. Pooling reduces the input processed by coarse blocks, while interpolation lets those blocks construct a long output from few coefficients.

The original experiments included horizons of 96, 192, 336, and 720. This does **not** mean that a model trained with $H=48$ can inherently forecast thousands of points. Its output structure is trained for 48 points. Covering a longer period requires repeated external calls or retraining with a longer horizon.

## 8. Relevance to electricity demand

Electricity demand contains structure at several scales:

$$
y_t=T_t+S_t^{\text{annual}}+S_t^{\text{weekly}}
+S_t^{\text{daily}}+\varepsilon_t.
$$

| Demand characteristic | Relevant N-HiTS mechanism |
|---|---|
| Slow changes in demand level | Coarse blocks |
| Weekly workday/weekend behaviour | Intermediate resolution |
| Daily consumption profile | Intermediate and fine blocks |
| Half-hourly peaks and troughs | Fine blocks |
| Long output sequence | Hierarchical interpolation |
| High-frequency history | Multi-rate pooling |

This makes N-HiTS a reasonable model to **test** on AEMO demand, but it does not prove that it will outperform every baseline.

## 9. Performance evidence motivating its use

This section presents empirical evidence rather than a technical model-by-model comparison.

### 9.1 Original N-HiTS study

The original paper compared N-HiTS with N-BEATS, DilRNN, auto-ARIMA, FEDformer, Autoformer, Informer, Reformer, and LogTrans. Across its multivariate datasets and horizons, the authors reported an average relative reduction of **14% in MAE** and **16% in MSE** against the best compared baseline. At the longest evaluated horizons, the reductions were **11% in MAE** and **17% in MSE**.

The paper has more than 1,000 citations in the scholarly index checked for this report. Citation counts change over time and differ between databases.

### 9.2 Real electricity-consumption results

The paper evaluated the Electricity Consumption Load (ECL) dataset, containing electricity consumption from 321 customers. Its reported normalised MAE results were:

| Horizon | N-HiTS | N-BEATS | FEDformer | Autoformer | Informer | Auto-ARIMA |
|---:|---:|---:|---:|---:|---:|---:|
| 96 | 0.249 | **0.247** | 0.297 | 0.317 | 0.368 | 0.814 |
| 192 | **0.269** | 0.283 | 0.308 | 0.334 | 0.386 | 0.842 |
| 336 | **0.290** | 0.308 | 0.313 | 0.338 | 0.394 | 0.866 |
| 720 | **0.340** | 0.362 | 0.343 | 0.361 | 0.439 | 0.891 |

N-BEATS was marginally better at $H=96$. N-HiTS achieved the lowest reported MAE at $H=192$, $336$, and $720$, and substantially outperformed Informer and auto-ARIMA at every listed horizon. This directly motivates an electricity-demand experiment.

The study also used ETTm2, containing real electricity-transformer measurements. N-HiTS achieved the lowest reported MSE and MAE among the listed methods at horizons 96, 192, 336, and 720 in the paper's main table.

### 9.3 Independent evidence: TFB

The independent **TFB: Towards Comprehensive and Fair Benchmarking of Time Series Forecasting Methods** paper has more than 200 citations in the index checked for this report. It compares 22 methods through a unified pipeline, including ARIMA, ETS, VAR, XGBoost, linear regression, random forest, RNN, N-HiTS, N-BEATS, Informer, FEDformer, and PatchTST.

TFB reports that N-HiTS, PatchTST, and TimesNet achieved substantially better **average** performance on univariate datasets under MASE and MSMAPE than many alternatives. It also finds that no model is best on every dataset and that simple machine-learning methods win more individual datasets in some comparisons.

This gives a balanced motivation:

> A later independent benchmark also identified N-HiTS as one of the stronger average univariate forecasters, but confirmed that performance must be tested on the target dataset.

The defensible justification for the AEMO project is therefore:

> N-HiTS has demonstrated strong long-horizon performance against Transformer, recurrent, and autoregressive statistical baselines, including on real electricity data. This justifies evaluating it as an advanced AEMO forecaster, but does not guarantee superiority under the project's forecasting protocol.

## 10. AEMO configuration

The implementation wraps Nixtla's `NeuralForecast` API behind the project's `Forecaster` interface (`NHITSForecaster`). The horizon and look-back are

$$
H = 336,\qquad L = 672,
$$

so the model predicts one week from the previous two weeks of half-hourly demand.

| Parameter | Value | Meaning |
|---|---:|---|
| Frequency | 30 minutes | one observation per half hour |
| `h` (horizon) | 336 | forecast the next 7 days |
| `input_size` (look-back) | 672 | condition on the previous 14 days |
| `scaler_type` | `robust` | per-window median / IQR normalisation |
| `max_steps` | 1000 | maximum optimiser updates |
| `learning_rate` | $10^{-3}$ | optimiser step size |
| `num_lr_decays` | 3 | scheduled learning-rate reductions |
| `val_size` | 672 | final 14 training days held out for early stopping |
| `early_stop_patience_steps` | 5 | stop after 5 validation checks without improvement |
| `val_check_steps` | 50 | validate every 50 steps |
| `random_seed` | 1 | reproducible initialisation and sampling |
| Accelerator | CPU | training hardware |

**Horizon choice.** The paper's benchmarks use hourly data, so its longest horizon $H=720$ is 30 days. On half-hourly AEMO data $H=720$ is only 15 days, and 15-day-ahead targets are dominated by the weekly cycle with few effectively-independent training windows in two years of one series, so the model would underfit toward a climatological average. $H=336$ (7 days) instead:

- matches the 7-day rolling-MAE window used throughout the project, so every degradation-curve point is a rolling mean over forecasts that are all at most one week ahead;
- is learnable from roughly two annual cycles of a single series;
- is long enough that a stale 2018–2019 relationship shows up in the error, but short enough that the output remains a genuine forecast rather than a seasonal template.

**Architecture.** Three identity stacks, one block each, two 512-unit hidden layers per block, max pooling, linear interpolation, ReLU, MAE loss, no dropout. The multi-rate factors are scaled for a weekly horizon rather than left at the API defaults (`[2, 2, 1]` / `[4, 2, 1]`, which suit short hourly horizons):

| Stack | Pool kernel | Frequency downsample | Knots over $H=336$ | Role |
|---|---:|---:|---:|---|
| Coarse | 16 | 48 | 7 | daily-level trend across the week |
| Intermediate | 8 | 24 | 14 | twice-daily / duck-curve shape |
| Fine | 1 | 1 | 336 | half-hourly detail |

**Scaler.** `robust` normalises each context window by its median and inter-quartile range before the forward pass and de-normalises the output afterwards. It is preferred over `identity` on raw megawatt magnitudes (about 1000–2000 MW), where an unnormalised network trains slowly and, under recursive feedback, can compound without bound. `identity` is appropriate only if the input series is standardised beforehand.

`max_steps=1000` with early stopping is a practical budget, not evidence of an optimum.

## 11. Training, calibration, and test periods

| Period | Role |
|---|---|
| 2018–2019, less the last 14 days | optimises N-HiTS weights |
| Last 14 days of 2019 | validation set for early stopping |
| January–February 2020 (calibration) | supplies known pre-test history without any weight update |
| March 2020 onward (test) | evaluates forecasts and degradation |

After training the parameters are frozen:

$$
\boldsymbol\Theta = \boldsymbol\Theta_{2018\text{--}2019}.
$$

Calibration observations are appended to the model's history buffer without invoking gradient descent. The first test forecast conditions on the most recent 672 observations available at that point — the tail of the calibration window:

$$
\hat{\mathbf y}_{1:336}
= f_{\boldsymbol\Theta_{2018\text{--}2019}}
\!\left(\mathbf y_{\text{calibration},\,-671:0}\right).
$$

Only the latest 14 days enter this forward pass, although the weights were learned from windows spanning the whole training period.

## 12. Why rolling forecasting is required

A single inference implements

$$
f_{\boldsymbol\Theta}:\mathbb R^{672}\rightarrow\mathbb R^{336}.
$$

The test period spans roughly four years, far more than 336 observations. The experiment partitions it into consecutive, non-overlapping 7-day blocks and calls the model once per block:

$$
\hat{\mathbf y}^{(k)} = f_{\boldsymbol\Theta}(\mathbf c_k),
$$

where $\mathbf c_k$ is the 672-point context at the start of block $k$.

This rolling procedure is **not part of the N-HiTS architecture**. It is an experiment-level policy for applying a fixed-horizon model across a multi-year test stream, and the choice of policy — described next — materially affects stability and comparability.

## 13. Rolling protocols and trade-offs

Two protocols are implemented and selected by one flag in `run_aemo_nhits.py`. An earlier "periodic re-grounding" variant (real data inserted on a fixed schedule, model forecasts used otherwise) was removed: it is neither of the two clean conditions, and it gave the model information the blind baselines did not receive.

### 13.1 Block (observed-context) forecasting — default

The model forecasts block $k$ from real recent history. After the block's forecasts are recorded and scored, the block's **actual** demand is revealed and appended to the history buffer as context for block $k+1$. The model never consumes its own forecasts, so nothing compounds. The issued forecast is never revised, and the weights are unchanged:

$$
\boldsymbol\Theta_{k+1} = \boldsymbol\Theta_k .
$$

Using newly-available history is not weight adaptation and not leakage: a real value only becomes context for a *later* block, never used to alter a block that was already issued. This is repeated one-week-ahead operation with a frozen model.

### 13.2 Fully blind forecasting

Test actuals are used only for error calculation. Each predicted block is appended to history:

$$
\mathbf c_2 = \bigl[\text{remaining calibration context},\ \hat{\mathbf y}^{(1)}\bigr],\qquad
\mathbf c_{k+1} = \bigl[\ldots,\ \hat{\mathbf y}^{(k)}\bigr].
$$

With $L=672$ and 7-day blocks the context is entirely model-generated after two blocks. The network was trained on real inputs but now receives its own predictions; bias accumulates, the input drifts outside the training distribution, and the forecast degrades. This satisfies a strict no-test-observation rule but changes the task from repeated one-week-ahead forecasting into an approximate multi-year extrapolation from a single origin.

### 13.3 Comparison

| Protocol | Question answered | Stability | Fair-comparison condition |
|---|---|---|---|
| Block (observed context) | Does a frozen one-week-ahead model degrade while current history remains available? | High | every model receives observations after each completed block |
| Fully blind | Can a frozen model extrapolate the whole test future without feedback? | Poor | no model receives test observations |

The project must fix the forecasting question first and apply the same information policy to every model. A frozen model means unchanged weights; it does not by itself require that all observations after the first origin be withheld.

### 13.4 Observed AEMO behaviour

Both protocols were run on SA1 and NSW1 (`split_id` `aemo_nhits_block7d_v1` and `aemo_nhits_blind_v1`; identical weights, `random_seed=1`).

| Protocol | Region | Overall MAE (MW) | 7-day rolling MAE, mean → max (MW) |
|---|---|---:|---:|
| Block | SA1 | 171 | 171 → 580 |
| Block | NSW1 | 506 | 506 → 1233 |
| Blind | SA1 | 7683 | 7638 → 34878 |
| Blind | NSW1 | 3645 | 3651 → 5400 |

For reference, the deterministic baselines on the same test window (all fully blind) score SA1 / NSW1 MAE of 156 / 442 (seasonal naive), 220 / 743 (XGBoost), and 237 / 880 (DHR + ARIMA).

**Block.** N-HiTS is second-best in both regions, behind seasonal naive. Its degradation is mild: the SA1 yearly-mean rolling MAE rises from about 155 (2020) to 179 (2023), roughly $+15\%$, against about $+40\%$ for blind XGBoost over the same span. Weekly re-anchoring on real data largely shields a one-week-ahead forecast from slow drift; the intra-year swing (summer peaks near 235, autumn troughs near 140) dominates the trend.

**Blind.** The forecast diverges. On SA1 the quarterly-mean rolling MAE climbs almost monotonically from 147 (Q1 2020) to about 34600 (start of 2024) — a factor of roughly 235 — and the body maximum of 34878 MW exceeds actual demand by an order of magnitude. NSW1 rises to roughly 4000–4800 MW and then saturates. Unlike an earlier attempt with `scaler_type="identity"` and $H=48$, which reached about $10^{37}$ MW and then `NaN`, this configuration stays finite: the robust per-window scaler keeps each output scaled to its context's own spread, and the 7-day blocks give about 200 recursive steps rather than about 1400. The divergence is slower and bounded, not removed.

The blind curve is a valid log — no `NaN`, no infinities — but the values are a divergence artefact, not a forecast error. Plotting it alongside the other models requires a capped or logarithmic axis.

## 14. Report-ready conclusion

> N-HiTS is a direct multi-horizon architecture combining multi-rate input sampling, hierarchical interpolation, and doubly residual stacking. Each block processes the unexplained history at a chosen resolution, reconstructs a backcast that is removed from later inputs, and contributes an interpolated partial forecast to the summed output, producing a learned coarse-to-fine representation suited to signals with structure at several temporal scales. The original study reported strong performance against Transformer, recurrent, and auto-ARIMA baselines, including superior longer-horizon MAE on real electricity-consumption data, and the independent TFB benchmark identified it as one of the stronger average univariate forecasters while confirming that no method dominates every dataset. These findings motivate evaluating N-HiTS on AEMO demand rather than assuming superiority.
>
> In the AEMO experiment N-HiTS conditions on the previous 14 days of half-hourly demand to predict the next 7 days. It is trained on 2018–2019, with the final two weeks held out for early stopping and the January–February 2020 calibration window supplying known context without any weight update. Because each inference covers only one week, an external rolling procedure applies the frozen model across the multi-year test stream. Under block forecasting — real history revealed after each completed week, model forecasts never fed back — N-HiTS is the second-best forecaster in both regions behind seasonal naive and degrades only mildly, because weekly re-anchoring shields a one-week-ahead forecast from slow drift. Under fully blind forecasting the model recursively consumes its own output and the error diverges by more than two orders of magnitude, though the robust scaler keeps it finite. The two curves answer different questions and are comparable only when the same information policy is applied to every baseline.


## References

1. Challu, C., Olivares, K. G., Oreshkin, B. N., Garza, F., Mergenthaler-Canseco, M., and Dubrawski, A. (2023). [*N-HiTS: Neural Hierarchical Interpolation for Time Series Forecasting*](https://ojs.aaai.org/index.php/AAAI/article/view/25854/25626). *Proceedings of the AAAI Conference on Artificial Intelligence*, 37(6), 6989–6997.
2. Qiu, X. et al. (2024). [*TFB: Towards Comprehensive and Fair Benchmarking of Time Series Forecasting Methods*](https://www.vldb.org/pvldb/vol17/p2363-hu.pdf). *Proceedings of the VLDB Endowment*, 17(9), 2363–2377.
3. Xu, Z., Zeng, A., and Xu, Q. (2024). [*FITS: Modeling Time Series with 10k Parameters*](https://proceedings.iclr.cc/paper_files/paper/2024/file/701251e1db4a2e4dd2ef23f5265d5936-Paper-Conference.pdf). *International Conference on Learning Representations*. This later, highly cited study includes N-HiTS as an established forecasting baseline and reinforces the need to compare strong neural models with simpler alternatives under a common protocol.
4. Nixtla. [*NeuralForecast: N-HiTS API documentation*](https://nixtlaverse.nixtla.io/neuralforecast/models.nhits.html).
